In [4]:


from pathlib import Path

import pandas as pd
from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.constants import proj_db_path
from nhs_waiting_lists.utils.xdg import XDGBasedir
from sqlalchemy import create_engine
from sqlalchemy import text, bindparam

project_root = Path(XDGBasedir.get_data_dir(__app_name__))

DB_PATH_CONSOLIDATED = project_root / proj_db_path / "nhs_consolidated.db"
DATA_DIR = "./data"

# conn = sqlite3.connect(DB_PATH)

conn = create_engine(f"sqlite:///{DB_PATH_CONSOLIDATED}")

In [5]:
# PROVIDER_CODES = ['R0B', 'RAJ', 'RTH', 'RTE', "RTF", "RWF", "RTE", "REF", "RWH", "R0B", "RVJ", "RHW", "RDU", "RH8",
#                   "RWY",
#                   "RXC", "RL4", "RDE", "RXK", "RXR", "RJ2", "RN5", "RHU", "RGN", "RWP", "RWD", "RAJ", 'RTF', 'RXC']
# # PROVIDER_COEDS = ('RAJ', 'RTH', 'RJZ', 'RH5', 'R0B', 'RTF', 'RXC')
# TREATMENT_CODES = (
#     'C_100',
#     'C_101',
#     'C_110',
#     'C_320',
#     'C_330',
#     'C_400',
#     'C_502',
#     'C_301'
# )

PROVIDER_CODES = ["RTF", "RWF", "RTE", "REF", "RWH", "R0B", "RVJ", "RHW", "RDU", "RH8", "RWY", "RXC", "RL4", "RDE",
                  "RXK", "RXR", "RJ2", "RN5", "RHU", "RGN", "RWP", "RWD", "RAJ"]
# TREATMENT_CODES = ('C_101', 'C_110', 'C_320', 'C_330', 'C_400', 'C_502', 'C_301', 'C_100', 'C_101', 'C_110', 'C_120',
#                    'C_130', 'C_140', 'C_150', 'C_160', 'C_170', 'C_300', 'C_301', 'C_320', 'C_330', 'C_340', 'C_400',
#                    'C_410', 'C_430', 'C_502')

TREATMENT_CODES = ('C_999',)

provider_code = "RAJ"
treatment_code = "C_110"

In [6]:
consolidated_query = text("""
                          SELECT period,
                                 provider_code                                    as provider,
                                 treatment_function_code                          as treatment_code,
                                 i.incomplete_prev,
                                 i.incomplete - i.incomplete_prev                 AS inc_diff,
                                 (
                                     (i.incomplete / NULLIF(i.incomplete_prev, 0)
                                         ) - 1)                                   AS frac_change,
                                 -- the accounting identity
                                 (
                                     i.incomplete
                                         - (i.incomplete_prev
                                                + i.new_periods
                                         - i.nonadmitted
                                         - i.admitted
                                         ))                                       AS residual,
                                 -- scaled accounting identity
                                 (
                                     i.incomplete
                                         - (i.incomplete_prev
                                                + i.new_periods
                                         - i.nonadmitted
                                         - i.admitted
                                         )) / (i.incomplete_prev + i.new_periods) AS r,
                                 i.admitted,
                                 i.admitted_prev,
                                 i.nonadmitted,
                                 i.nonadmitted_prev,
                                 i.new_periods,
                                 i.new_periods_prev,
                                 i.incomplete
                          FROM metrics AS i
                          WHERE provider IN :provider_codes
                            AND treatment_code IN :treatment_codes
                            AND period >= '2025-08-01'
                          ORDER BY provider_code ASC, period ASC; \
                          """).bindparams(
    bindparam('provider_codes', expanding=True),
    bindparam('treatment_codes', expanding=True)
)

consolidated_df = pd.read_sql(consolidated_query, conn, params={
    'provider_codes': PROVIDER_CODES,
    'treatment_codes': TREATMENT_CODES}
                              )  # type: ignore[arg-type]

consolidated_df["period"] = pd.to_datetime(consolidated_df["period"], errors="coerce")

valid = (consolidated_df['incomplete_prev'] > 100) & (consolidated_df['new_periods'] > 30)
consolidated_df = consolidated_df.loc[valid].copy()
consolidated_df

,period,provider,treatment_code,incomplete_prev,inc_diff,frac_change,residual,r,admitted,admitted_prev,nonadmitted,nonadmitted_prev,new_periods,new_periods_prev,incomplete
0,2025-08-01,R0B,C_999,59131.0,113.0,0.001911,-2231.0,-0.029010,2916.0,3439.0,12514.0,14140.0,17774.0,20951.0,59244.0
1,2025-08-01,RAJ,C_999,173767.0,2949.0,0.016971,-5879.0,-0.028747,3808.0,4247.0,18104.0,22653.0,30740.0,37561.0,176716.0
2,2025-08-01,RDE,C_999,92240.0,1036.0,0.011232,-837.0,-0.007745,2074.0,2832.0,11878.0,15035.0,15825.0,22028.0,93276.0
3,2025-08-01,RDU,C_999,71588.0,-550.0,-0.007683,-1345.0,-0.015712,2438.0,3204.0,10785.0,13902.0,14018.0,16874.0,71038.0
4,2025-08-01,REF,C_999,40859.0,385.0,0.009423,-2155.0,-0.040139,2465.0,2833.0,7824.0,9773.0,12829.0,15207.0,41244.0
5,2025-08-01,RGN,C_999,78896.0,-891.0,-0.011293,-2362.0,-0.025420,1931.0,2267.0,10621.0,13343.0,14023.0,17621.0,78005.0
6,2025-08-01,RH8,C_999,74801.0,1109.0,0.014826,-955.0,-0.010365,4007.0,4670.0,11266.0,13924.0,17337.0,20373.0,75910.0
7,2025-08-01,RHU,C_999,58799.0,-1310.0,-0.022279,-3092.0,-0.043407,1902.0,2141.0,8750.0,10833.0,12434.0,15202.0,57489.0
8,2025-08-01,RHW,C_999,36763.0,-1755.0,-0.047738,-5570.0,-0.124997,1327.0,1389.0,2656.0,3168.0,7798.0,10929.0,35008.0
9,2025-08-01,RJ2,C_999,55547.0,-2014.0,-0.036258,-4903.0,-0.072574,1157.0,1045.0,7966.0,8914.0,12012.0,13766.0,53533.0


In [ ]:
import numpy as np

# After loading the dataframe
consolidated_df['r_peer_median'] = consolidated_df.groupby(
    ['period', 'treatment_code']
)['r'].transform('median')

consolidated_df['r_peer_mean'] = consolidated_df.groupby(
    ['period', 'treatment_code']
)['r'].transform('mean')

consolidated_df['r_peer_std'] = consolidated_df.groupby(
    ['period', 'treatment_code']
)['r'].transform('std')

group = consolidated_df.groupby(['period', 'treatment_code'])
consolidated_df['r_mad'] = group['r'].transform(lambda x: 1.4826 * (x - x.median()).abs().median())
consolidated_df['r_z'] = (consolidated_df['r'] - consolidated_df['r_peer_median']) / consolidated_df['r_mad']

consolidated_df['excess_flag'] = (consolidated_df['r_z'] < -2).astype(int)
consolidated_df['cum_excess'] = consolidated_df.groupby(['provider', 'treatment_code'])['excess_flag'].cumsum()

consolidated_df['denom'] = consolidated_df['incomplete_prev'] + consolidated_df['new_periods']
consolidated_df['r_low'] = consolidated_df['r_peer_median'] - 2 * consolidated_df['r_mad']
consolidated_df['expected_min_resid'] = consolidated_df['r_low'] * consolidated_df['denom']

# Missing patients this month (beyond expected min)
consolidated_df['excess_missing'] = np.where(consolidated_df['residual'] < consolidated_df['expected_min_resid'],
                                             consolidated_df['expected_min_resid'] - consolidated_df['residual'],
                                             0)

consolidated_df

In [ ]:
df_xs = consolidated_df.query("excess_missing > 0")

print(df_xs["excess_missing"].sum())

consolidated_df.query("provider == 'RXR' and treatment_code == 'C_110' and period == '2021-10-01'")[[
    'incomplete_prev', 'incomplete', 'new_periods', 'admitted', 'nonadmitted', 'residual', 'excess_missing'
]]


In [ ]:
provider_treatment_rank = (
    consolidated_df
    .groupby(['provider', 'treatment_code'], as_index=False)['excess_missing']
    .sum()
    .rename(columns={'excess_missing': 'total_excess_missing'})
)

# rank within treatment, or overall
provider_treatment_rank['rank_within_treatment'] = (
    provider_treatment_rank.groupby('treatment_code')['total_excess_missing']
    .rank(ascending=False, method='min')
)
provider_treatment_rank['rank_overall'] = provider_treatment_rank['total_excess_missing'] \
    .rank(ascending=False, method='min')

provider_treatment_rank = provider_treatment_rank.sort_values('total_excess_missing', ascending=False)

provider_treatment_rank.head(10)

In [ ]:
provider_rank = (
    consolidated_df
    .groupby(['provider'], as_index=False)['excess_missing']
    .sum()
    .rename(columns={'excess_missing': 'total_excess_missing'})
)
provider_rank = provider_rank.sort_values('total_excess_missing', ascending=False)

provider_rank.head(10)

In [ ]:
import matplotlib.pyplot as plt

top20 = provider_treatment_rank.head(20)
plt.figure(figsize=(10, 6))
plt.barh(top20['provider'], top20['total_excess_missing'])
plt.gca().invert_yaxis()
plt.xlabel('Excess missing patients (beyond peer norm)')
plt.title('Top-20 providers by cumulative “dark backlog”')
plt.tight_layout()
plt.show()

In [ ]:

from plotnine import (
    ggplot, aes, geom_histogram, geom_vline, labs
)

quantile_05 = consolidated_df["r_peer_median"].quantile(0.05)

apple_returns_figure = (
        ggplot(consolidated_df, aes(x="r_peer_median"))
        + geom_histogram(bins=100)
        + geom_vline(aes(xintercept=quantile_05), linetype="dashed")
        + labs(x="", y="", title="Distribution of daily Apple stock returns")

)
apple_returns_figure.show()

In [ ]:

from plotnine import (
    ggplot, aes, geom_histogram, geom_vline, labs
)

quantile_05 = consolidated_df["r"].quantile(0.05)

apple_returns_figure = (
        ggplot(consolidated_df, aes(x="r"))
        + geom_histogram(bins=100)
        + geom_vline(aes(xintercept=quantile_05), linetype="dashed")
        + labs(x="", y="", title="Distribution of daily Apple stock returns")

)
apple_returns_figure.show()

In [ ]:
from plotnine import (
    theme_minimal,
    scale_x_datetime
)
from plotnine import ggplot, aes, geom_line, facet_wrap, labs, theme_bw

p7 = (
        ggplot(consolidated_df, aes(x="period", y="incomplete", color="treatment_code", group="treatment_code"))
        + geom_line()
        + facet_wrap("~provider", ncol=2, scales="free_y")  # one plot per provider
        + labs(
    title="Incomplete Pathways by Specialty and Provider",
    x="Month",
    y="Total incomplete pathways",
    color="Specialty Code"
)
        + theme_bw(base_size=9)
)
p7

In [ ]:
p = (
        ggplot(consolidated_df.query(f"provider == '{provider_code}'"),
               aes(x="period", y="incomplete", color="treatment_code", group="treatment_code"))
        + geom_line()
        + theme_minimal()
        + scale_x_datetime(date_labels="%y-%m")
)
p

In [ ]:
p2 = (
        ggplot(consolidated_df.query(f"provider == '{provider_code}'").query("treatment_code == 'C_320'"),
               aes(x="period", y="incomplete", color="treatment_code", group="treatment_code"))
        + geom_line()
        + theme_minimal()
        + scale_x_datetime(date_labels="%y-%m")
)
p2

In [ ]:

import statsmodels.api as sm

df = consolidated_df.query(f"provider == '{provider_code}'").query("treatment_code == 'C_320'")
X = sm.add_constant(df['r_peer_median'])
y = df['r']
ols = sm.OLS(y, X).fit(cov_type='HC3')
print(ols.params, ols.rsquared)
print(ols.summary2())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# # Convert to datetime if not already
# subset_c_400 = consolidated_df[(consolidated_df["provider"] == "RAJ") & (consolidated_df["treatment_code"] == "C_400")].copy()
# subset_c_400["period"] = pd.to_datetime(subset_c_400["period"], errors="coerce")

consolidated_df = consolidated_df.reset_index()

returns_wide = (
    consolidated_df[(consolidated_df["provider"] == provider_code)].query(
        'treatment_code == "C_400" or treatment_code=="C_320"')
    .pivot(
        index="period",
        columns="treatment_code",
        values="residual"
    )
    .reset_index()
)
returns_wide

In [ ]:


# Plot
fig, ax = plt.subplots(
    figsize=(10, 15),
    sharex=True,
    sharey=False,
    nrows=3,
    ncols=1,
)
returns_wide["period"] = pd.to_datetime(returns_wide["period"], errors="coerce")


# subset_c_400 = df[(df["provider"] == "RAJ") & (df["treatment_code"] == "C_400")].copy()
# subset_c_400["period"] = pd.to_datetime(subset_c_400["period"], errors="coerce")

def align_yaxis_zero(ax1, ax2):
    """Align the zero points of two y-axes."""
    y1_min, y1_max = ax1.get_ylim()
    y2_min, y2_max = ax2.get_ylim()

    # Calculate the ratio of positive to negative range for ax1
    if y1_min < 0 < y1_max:
        ratio1 = y1_max / abs(y1_min)
    else:
        return  # Don't adjust if ax1 doesn't cross zero

    # Apply same ratio to ax2 if it crosses zero
    if y2_min < 0 < y2_max:
        # Adjust limits to match ratio
        new_y2_max = abs(y2_min) * ratio1
        ax2.set_ylim(y2_min, new_y2_max)


for ax_idx, colname in ([0, "C_320"], [1, "C_400"], [2, "C_110"]):
    ax[ax_idx].grid(True, alpha=0.3)

    subset = consolidated_df.query(f"provider == '{provider_code}' and treatment_code == @colname")

    ax[ax_idx].plot(subset["period"], subset["incomplete"], lw=1.5)
    ax[ax_idx].set_title(f"Residual Flow {colname}")
    ax[ax_idx].set_ylabel("Residual patients")
    ax[ax_idx].grid(True, alpha=0.3)
    ax[ax_idx].fill_between(subset["period"], subset["r"], where=subset["r"] < 0, facecolor='green', alpha=.5)
    ax[ax_idx].plot(subset["period"], subset["residual"], lw=1.5)

    ax2 = ax[ax_idx].twinx()
    ax2.plot(subset["period"], subset["r"], lw=1.5)
    ax2.fill_between(subset["period"], subset["r"], where=subset["r"] < 0, facecolor='green', alpha=.5)
    align_yaxis_zero(ax[ax_idx], ax2)

    # Yearly major ticks
    ax[ax_idx].xaxis.set_major_locator(mdates.YearLocator())
    ax[ax_idx].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # # Optional: quarterly minor ticks
    # ax[0].xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    # ax[0].xaxis.set_minor_formatter(mdates.DateFormatter("%b"))

    plt.setp(ax[ax_idx].get_xticklabels(), rotation=0, ha="center")
plt.tight_layout()
plt.show()


In [ ]:

treatments = subset["treatment_code"].unique()
ncols = 3
nrows = -(-len(treatments) // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows), sharex=True)
axes = axes.flatten()

for ax, tcode in zip(axes, treatments):
    d = subset[subset["treatment_code"] == tcode]
    d = d.sort_values("period")

    ax.xaxis.set_major_locator(mdates.YearLocator())  # one tick per year
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis="x", rotation=0)

    d_norm = d[["admitted", "nonadmitted", "new_periods"]].div(
        d[["admitted", "nonadmitted", "new_periods"]].sum(axis=1), axis=0)

    ax.stackplot(
        d["period"],
        d_norm["new_periods"],
        d_norm["admitted"],
        d_norm["nonadmitted"],
        # labels=["new_periods", "Admitted", "Non-admitted"],
        alpha=0.8,
    )
    ax.set_title(tcode)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)

# tidy up
for ax in axes[len(treatments):]:
    ax.set_visible(False)

fig.suptitle("RAJ – RTT Flows by Specialty", fontsize=16)
fig.tight_layout()
fig.subplots_adjust(top=0.93)

# one global legend
fig.legend(["New periods", "Admitted", "Non-admitted"],
           loc="lower right", bbox_to_anchor=(0.98, 0.95))

plt.show()
